## Designing binders for S-100B with BoltzGen

In [1]:
!pip install boltzgen pandas -q

### 1. Download S100B target structure

Using PDB entry **3D0Y** — S100B in the Ca²⁺/Zn²⁺-loaded state. S100B is a **homodimer**: chains A and B together form the functional binding surfaces (hydrophobic cleft, RAGE-binding face). We keep both chains and strip only the HETATM metal ions.

https://www.rcsb.org/structure/3D0Y (PDB)

binding site/hotspot: https://pmc.ncbi.nlm.nih.gov/articles/PMC6405260/

In [2]:
import urllib.request
import os

os.makedirs("s100b_binder", exist_ok=True)

# Official RCSB PDB download
urllib.request.urlretrieve(
    "https://files.rcsb.org/download/3D0Y.pdb",
    "s100b_binder/3D0Y.pdb"
)

# Keep ATOM lines for chains A and B only; discard HETATM (Ca2+, Zn2+)
clean = []
with open("s100b_binder/3D0Y.pdb") as f:
    for line in f:
        if line.startswith("ATOM") and line[21] in ("A", "B"):
            clean.append(line)

target_pdb = "s100b_binder/s100b_dimer.pdb"
with open(target_pdb, "w") as f:
    f.writelines(clean)
    f.write("END\n")

chains = set(l[21] for l in clean)
print(f"Saved {len(clean)} ATOM lines, chains: {sorted(chains)}")
print("First residue:", clean[0][17:26].strip(), "  Last residue:", clean[-1][17:26].strip())

Saved 1471 ATOM lines, chains: ['A', 'B']
First residue: SER A   1   Last residue: GLU B  89


In [3]:
!pip install py3Dmol -q

In [4]:
import py3Dmol

with open("s100b_binder/s100b_dimer.pdb") as f:
    pdb_str = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_str, "pdb")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "cyan"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "salmon"}})
view.setBackgroundColor("white")
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### 2. Write BoltzGen design specification

The YAML spec tells BoltzGen:
- **target**: the S100B chain to bind (chain A, all residues)
- **binder**: a new protein chain of 60–100 residues to design de novo

S100B's known binding hotspot is the hydrophobic cleft exposed on Ca²⁺-binding (roughly residues 36–51, 69–83). We leave `hotspot` unset here to let BoltzGen find the interface, but you can pin residues with `hotspot_residues` to focus the search.

In [5]:
design_yaml = """
entities:
  # de novo binder, 60–100 residues (length randomly sampled in range)
  - protein:
      id: C
      sequence: 60..100
  # target: S100B homodimer, chains A and B from the PDB
  - file:
      path: s100b_dimer.pdb        # relative to THIS yaml's directory
      include:
        - chain:
            id: A
        - chain:
            id: B
"""

spec_path = "s100b_binder/design_spec.yaml"
with open(spec_path, "w") as f:
    f.write(design_yaml)
print(f"Wrote design spec to {spec_path}")

Wrote design spec to s100b_binder/design_spec.yaml


### 3. Run BoltzGen

`boltzgen run` runs the full pipeline:
1. **Design** — diffusion model generates binder backbone conformations
2. **Inverse folding** (BoltzIF) — assigns sequences to each backbone
3. **Validation** — refolds each sequence with the target to confirm binding
4. **Scoring** — ranks designs by predicted interface quality

`--recycling-steps 3` and `--diffusion-samples 5` control accuracy vs. speed. For a quick test, use `--n-designs 10`; scale to 50–100 for a real campaign (~30–60 s/design on a single GPU).

In [7]:
  !boltzgen run s100b_binder/design_spec.yaml \
      --output=s100b_binder/output \
      --num_designs=2 \
      --use_kernels false


=== Configuring pipeline ===
Using dataset artifact: /home/bridget/.cache/huggingface/hub/datasets--boltzgen--inference-data/snapshots/c3d36fd276e9caf098c75d4113c6d5eb320b1a4c/mols.zip
************** Checking design spec: s100b_binder/design_spec.yaml **************
Total designed residues: 66
There are 0 unresolved residues and 20 unresolved atoms in the target.
Design specification visualization is written to s100b_binder/output/design_spec.cif
*********************************************************************************
Using kernels: False [device capability: (8, 6)]
Config overrides for protocol protein-anything: {}
Using 1 devices
Raw designs will be saved to: s100b_binder/output/intermediate_designs
Using diffusion batch size: 1
Number of diffusion batches: 2
Using model artifact: /home/bridget/.cache/huggingface/hub/models--boltzgen--boltzgen-1/snapshots/c1be29e1f82ffcc72264f64b993c43fb4e0d17f0/boltzgen1_diverse.ckpt
Using model artifact: /home/bridget/.cache/huggingface/h

### 4. Inspect and filter results

BoltzGen writes one subdirectory per design under `output/`. Each contains:
- `complex.cif` — predicted complex structure
- `binder.fasta` — designed binder sequence
- `scores.json` — per-design metrics

Key metrics to filter on:
| Metric | Meaning | Keep if |
|--------|---------|---------|
| `ipTM` | interface predicted TM-score (0–1) | ≥ 0.6 |
| `pTM` | overall pTM of the complex | ≥ 0.5 |
| `pLDDT_binder` | per-residue confidence of binder | ≥ 70 |
| `binding_energy` | predicted ΔG (kcal/mol, lower = better) | depends on target |

In [8]:
import pandas as pd, glob

out = "s100b_binder/output/final_ranked_designs"

all_metrics = pd.read_csv(f"{out}/all_designs_metrics.csv")
print(f"{len(all_metrics)} designs scored")
print("Columns:", list(all_metrics.columns))   # see the real metric + sequence column names

final_csv = glob.glob(f"{out}/final_designs_metrics_*.csv")
final = pd.read_csv(final_csv[0]) if final_csv else all_metrics
final.head(20)

2 designs scored
Columns: ['id', 'final_rank', 'designed_sequence', 'designed_chain_sequence', 'num_design', 'design_to_target_iptm', 'min_design_to_target_pae', 'design_ptm', 'filter_rmsd', 'designfolding-filter_rmsd', 'plip_hbonds_refolded', 'delta_sasa_refolded', 'design_largest_hydrophobic_patch_refolded', 'design_chain_hydrophobicity', 'design_hydrophobicity', 'loop', 'helix', 'sheet', 'file_name', 'num_prot_tokens', 'num_lig_atoms', 'num_resolved_tokens', 'num_tokens', 'UNK_fraction', 'GLY_fraction', 'ALA_fraction', 'CYS_fraction', 'SER_fraction', 'PRO_fraction', 'THR_fraction', 'VAL_fraction', 'ILE_fraction', 'ASN_fraction', 'ASP_fraction', 'LEU_fraction', 'MET_fraction', 'GLN_fraction', 'GLU_fraction', 'LYS_fraction', 'HIS_fraction', 'PHE_fraction', 'ARG_fraction', 'TYR_fraction', 'TRP_fraction', 'liability_score', 'liability_num_violations', 'liability_high_severity_violations', 'liability_medium_severity_violations', 'liability_low_severity_violations', 'liability_ProtTryp_co

,id,final_rank,designed_sequence,designed_chain_sequence,num_design,design_to_target_iptm,min_design_to_target_pae,design_ptm,filter_rmsd,designfolding-filter_rmsd,...,rank_design_to_target_iptm,rank_design_ptm,rank_neg_min_design_to_target_pae,rank_plip_hbonds_refolded,rank_plip_saltbridge_refolded,rank_delta_sasa_refolded,max_rank,secondary_rank,quality_score,sequence
0,design_spec_1,1,MGAAGLAMVVAAERAEEAGRKGITVEDVEYGARKAKEVDPDFDVEG...,MGAAGLAMVVAAERAEEAGRKGITVEDVEYGARKAKEVDPDFDVEG...,98,0.45703,5.24300,0.65005,1.06869,13.06670,...,1.0,1.0,1.0,0.5,0.5,0.5,1.0,1,1.0,MGAAGLAMVVAAERAEEAGRKGITVEDVEYGARKAKEVDPDFDVEG...
1,design_spec_0,2,SLTEEQMKEAGKLASAVIIAVYCLKKGKSKEEIKKEIEKYFGPLTD...,SLTEEQMKEAGKLASAVIIAVYCLKKGKSKEEIKKEIEKYFGPLTD...,91,0.29560,8.52023,0.64641,18.76188,11.70698,...,2.0,2.0,2.0,1.0,1.0,1.0,2.0,2,0.0,SLTEEQMKEAGKLASAVIIAVYCLKKGKSKEEIKKEIEKYFGPLTD...


In [9]:
import glob, os, py3Dmol

out = "s100b_binder/output"
TARGET_CHAINS = {"A", "B"}

# Panel 0 = apo target; panels 1..N = designs (target+binder complexes)
apo = open("s100b_binder/s100b_dimer.pdb").read()
panels = [("S100B dimer (apo — no binder)", apo, "pdb")]

cif_paths = sorted(
    glob.glob(f"{out}/final_ranked_designs/final_*_designs/*.cif")
    or glob.glob(f"{out}/final_ranked_designs/intermediate_ranked_*_designs/*.cif")
)
for p in cif_paths:
    panels.append((os.path.basename(p).replace(".cif", ""), open(p).read(), "cif"))

print(f"{len(panels)} panels: 1 apo target + {len(cif_paths)} designs")

ncols = min(len(panels), 3)
nrows = (len(panels) + ncols - 1) // ncols
view = py3Dmol.view(width=400 * ncols, height=340 * nrows, viewergrid=(nrows, ncols))

for i, (name, text, fmt) in enumerate(panels):
    r, c = divmod(i, ncols)
    view.addModel(text, fmt, viewer=(r, c))
    view.setBackgroundColor("white", viewer=(r, c))
    if i == 0:
        # apo target: all gray (chains faintly distinct so the dimer reads)
        view.setStyle({"chain": "A"}, {"cartoon": {"color": "lightgrey"}}, viewer=(r, c))
        view.setStyle({"chain": "B"}, {"cartoon": {"color": "grey"}},      viewer=(r, c))
    else:
        # design: everything magenta (binder), then recolor target chains gray
        view.setStyle({}, {"cartoon": {"color": "magenta"}}, viewer=(r, c))
        for ch in TARGET_CHAINS:
            view.setStyle({"chain": ch}, {"cartoon": {"color": "lightgrey"}}, viewer=(r, c))
    view.addLabel(name, {"fontSize": 11, "backgroundOpacity": 0.6,
                         "alignment": "topCenter"}, viewer=(r, c))
    view.zoomTo(viewer=(r, c))

view.show()

3 panels: 1 apo target + 2 designs


3Dmol.js failed to load for some reason. Please check your browser console for error messages.